# PTCG AI Battle — Public Research Bootstrap

This notebook builds a read-only research index from a Kaggle Dataset containing public 2026 agent repositories and the competition-authorized inputs. It does not execute third-party agent code automatically.

**Research goal:** identify reusable architectures, replay-processing methods, model artifacts, deck lists, evaluation tools and failure findings before we train our own agent.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/ptcg_research_index')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Kaggle inputs:', [p.name for p in INPUT_ROOT.iterdir()])

## 1. Locate the attached research dataset

The notebook accepts either a prepared dataset with `repos/` snapshots or an empty input followed by a `git clone` bootstrap when Internet is enabled.

In [ ]:
candidates = [p for p in INPUT_ROOT.iterdir() if p.is_dir()]
for p in candidates:
    if (p / 'repos').exists() or (p / 'public_research_sources.csv').exists():
        print('Research dataset candidate:', p)

research_input = next((p for p in candidates if (p / 'repos').exists()), None)
print('Selected:', research_input)

## 2. Optional online bootstrap

Enable this only if Kaggle Internet is available. It clones the exact URLs in the manifest into the working directory and never imports or runs their code.

In [ ]:
REPOS = {
  'scio': 'https://github.com/SiamRahman29/pokemon-tcg-agent.git',
  'lawliet': 'https://github.com/Lawliet-ai/ptcg-ai-battle-agent.git',
  'ptcg_abc': 'https://github.com/wmh/ptcg-abc.git',
  'ducdata': 'https://github.com/Ducdata1808/PTCG_Agent.git',
  'pckhoa': 'https://github.com/pckhoa206/kaggle_challenge_pokemon.git',
  'scha54': 'https://github.com/scha54/Kaggle-Pokemon.git',
}
online_root = WORK_ROOT / 'cloned_repos'
online_root.mkdir(exist_ok=True)

ENABLE_ONLINE_CLONE = False  # set True only when Internet is enabled
if ENABLE_ONLINE_CLONE:
    for name, url in REPOS.items():
        dest = online_root / name
        if not dest.exists():
            subprocess.run(['git', 'clone', '--depth', '1', url, str(dest)], check=True)
print('Online root:', online_root)

## 3. Inventory repositories, notebooks, models and decks

In [ ]:
roots = []
if research_input is not None:
    roots.append(research_input / 'repos')
if online_root.exists():
    roots.append(online_root)
roots = [r for r in roots if r.exists()]

MODEL_EXT = {'.pt','.pth','.onnx','.npz','.ckpt','.safetensors','.bin','.pkl','.joblib'}
rows = []
for root in roots:
    for f in root.rglob('*'):
        if not f.is_file() or '.git' in f.parts:
            continue
        ext = f.suffix.lower()
        kind = 'model' if ext in MODEL_EXT else 'notebook' if ext == '.ipynb' else 'deck' if f.name.lower() in {'deck.csv','decklist.csv'} else 'file'
        rows.append({'repo_root': str(root), 'path': str(f), 'kind': kind, 'bytes': f.stat().st_size})

out_csv = WORK_ROOT / 'research_artifacts.csv'
with out_csv.open('w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=['repo_root','path','kind','bytes'])
    w.writeheader(); w.writerows(rows)

print('Artifacts:', len(rows))
for kind in sorted({r['kind'] for r in rows}):
    print(kind, sum(r['kind']==kind for r in rows))

## 4. Extract only what we need for our own training

Do **not** train on every file blindly. The next stage should explicitly select:

- high-quality replay datasets / state-action examples;
- model architectures and feature encoders;
- deck lists and archetype labels;
- evaluation/arena utilities;
- experiment findings and known failure modes.

This keeps provenance and lets us compare every imported idea against our exact Playground simulator.

In [ ]:
import pandas as pd
artifact_df = pd.DataFrame(rows)
artifact_df.head(20)

## 5. Produce normalized research outputs

At the end of the notebook we keep a small index that can be attached to later experiments, while large source snapshots remain in the Kaggle Dataset input.

In [ ]:
summary = {
  'artifact_count': len(artifact_df),
  'models': int((artifact_df.kind == 'model').sum()) if len(artifact_df) else 0,
  'notebooks': int((artifact_df.kind == 'notebook').sum()) if len(artifact_df) else 0,
  'decks': int((artifact_df.kind == 'deck').sum()) if len(artifact_df) else 0,
}
(WORK_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(summary)